# Black-Litterman Crypto — Exploração e Validação

Este notebook valida os módulos implementados e demonstra o modelo end-to-end.

**Estrutura:**
1. Carregar dados
2. Estatísticas descritivas dos retornos
3. Heatmap de correlação
4. Retornos acumulados (base 100)
5. Pesos de mercado atuais
6. Retornos implícitos de equilíbrio (Π)
7. Comparação Π vs retornos históricos
8. BL sem views → deve recuperar pesos de mercado
9. BL com views RSI → como pesos mudam
10. Plot comparativo de pesos

In [1]:
import sys
from pathlib import Path

# Garante que src/ seja encontrado ao rodar de dentro de notebooks/
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.config import ARQUIVO_PRECOS, ARQUIVO_RETORNOS, ARQUIVO_MARKET_CAP, CATEGORIAS
from src.portfolio_utils import (
    calcular_matriz_covariancia,
    calcular_volatilidades,
    estatisticas_portfolio,
    aplicar_pesos,
    DIAS_ANO_CRIPTO,
)
from src.views import gerar_views_rsi
from src.black_litterman import BlackLitterman

sns.set_theme(style='whitegrid', palette='tab10')
pd.set_option('display.float_format', '{:.4f}'.format)
print('Setup OK')

Setup OK


## 1. Carregar dados

In [2]:
precos   = pd.read_parquet(ARQUIVO_PRECOS)
retornos = pd.read_parquet(ARQUIVO_RETORNOS)
mc_df    = pd.read_parquet(ARQUIVO_MARKET_CAP)

# Market cap indexado pelo ticker
mc_df = mc_df.set_index('ticker') if 'ticker' in mc_df.columns else mc_df
market_caps = mc_df['market_cap_usd'].rename('market_cap_usd')

# Alinha ativos presentes em todos os datasets
ativos = precos.columns.tolist()
retornos = retornos[ativos]
market_caps = market_caps.reindex(ativos).dropna()
ativos_validos = market_caps.index.tolist()

print(f'Período : {precos.index.min().date()} → {precos.index.max().date()}')
print(f'Ativos  : {ativos_validos}')
print(f'Shape   : {precos.shape}')

OSError: Repetition level histogram size mismatch

## 2. Estatísticas descritivas dos retornos

In [ ]:
fator = DIAS_ANO_CRIPTO

stats = pd.DataFrame({
    'retorno_anual_%' : retornos[ativos_validos].mean() * fator * 100,
    'vol_anual_%'     : retornos[ativos_validos].std()  * (fator**0.5) * 100,
    'sharpe'          : (retornos[ativos_validos].mean() * fator) / (retornos[ativos_validos].std() * fator**0.5),
    'min_diario_%'    : retornos[ativos_validos].min() * 100,
    'max_diario_%'    : retornos[ativos_validos].max() * 100,
    'obs'             : retornos[ativos_validos].count(),
    'categoria'       : pd.Series(CATEGORIAS),
}).round(2)

stats.sort_values('retorno_anual_%', ascending=False)

## 3. Heatmap de correlação

In [ ]:
corr = retornos[ativos_validos].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', vmin=-0.2, vmax=1,
    linewidths=0.5, ax=ax
)
ax.set_title('Correlação de Retornos Diários', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Retornos acumulados (base 100)

In [ ]:
precos_norm = precos[ativos_validos].div(precos[ativos_validos].iloc[0]) * 100

fig, ax = plt.subplots(figsize=(12, 5))
for ativo in ativos_validos:
    ax.plot(precos_norm.index, precos_norm[ativo], label=ativo, linewidth=1.2)

ax.axhline(100, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
ax.set_title('Retornos Acumulados (base 100)', fontsize=13)
ax.set_ylabel('Índice')
ax.legend(ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

## 5. Pesos de mercado atuais

In [ ]:
w_mkt = market_caps / market_caps.sum()
w_mkt_pct = (w_mkt * 100).round(2).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(w_mkt_pct.index, w_mkt_pct.values, color=sns.color_palette('tab10', len(w_mkt_pct)))
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_title('Pesos de Mercado (% market cap)', fontsize=13)
ax.set_ylabel('%')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.show()

print(w_mkt_pct.to_string())

## 6. Retornos implícitos de equilíbrio (Π)

In [ ]:
bl = BlackLitterman(
    retornos=retornos[ativos_validos],
    market_caps=market_caps,
    risk_aversion=2.5,
    tau=0.05,
)

pi = bl.calcular_retornos_implicitos()
print('Retornos implícitos de equilíbrio (anualizados):')
print((pi * 100).round(2).sort_values(ascending=False).to_string())

## 7. Π vs retornos históricos médios

O ponto central do Black-Litterman: os retornos implícitos **não são iguais** 
aos retornos históricos médios. Usar a média histórica diretamente leva a 
portfólios concentrados e instáveis.

In [ ]:
ret_historico = retornos[ativos_validos].mean() * DIAS_ANO_CRIPTO

comp = pd.DataFrame({
    'Histórico (μ)':     ret_historico * 100,
    'Implícito BL (Π)':  pi * 100,
}).round(2)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(comp))
w = 0.35
ax.bar(x - w/2, comp['Histórico (μ)'],    w, label='Histórico (μ)',    alpha=0.85)
ax.bar(x + w/2, comp['Implícito BL (Π)'], w, label='Implícito BL (Π)', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(comp.index, rotation=30)
ax.axhline(0, color='black', linewidth=0.7)
ax.set_title('Retorno Histórico vs Retorno Implícito BL (% a.a.)', fontsize=13)
ax.set_ylabel('%')
ax.legend()
plt.tight_layout()
plt.show()

print(comp)

## 8. BL sem views → deve aproximar pesos de mercado

Propriedade fundamental: otimizando com Π (sem views), a solução 
ótima deve ser exatamente w_mkt.

In [ ]:
resultado_eq = bl.executar()  # sem P, Q, Omega

comp_pesos = pd.DataFrame({
    'Mercado':   resultado_eq['pesos_mercado'] * 100,
    'BL s/views': resultado_eq['pesos_otimos'] * 100,
}).round(2)

print('Comparação de pesos (%):')
print(comp_pesos)
print(f"\nDiferença máxima: {(comp_pesos['Mercado'] - comp_pesos['BL s/views']).abs().max():.4f}%")

## 9. BL com views RSI

In [ ]:
data_ref = precos.index[-1]  # última data disponível

try:
    P, Q, Omega = gerar_views_rsi(
        precos=precos[ativos_validos],
        data_referencia=data_ref,
        threshold_compra=35,
        threshold_venda=65,
        retorno_esperado_view=0.05,
    )
    print(f'{len(Q)} view(s) RSI gerada(s) para {data_ref.date()}')
    print(f'Q (retornos esperados das views): {(Q * 100).round(1)}')
except ValueError as e:
    print(f'Nenhuma view ativa: {e}')
    P = Q = Omega = None

In [ ]:
if P is not None:
    resultado_views = bl.executar(P=P, Q=Q, Omega=Omega)
else:
    resultado_views = resultado_eq
    print('Usando resultado sem views (nenhuma view RSI ativa).')

## 10. Comparativo de pesos: mercado vs BL com views

In [ ]:
df_plot = pd.DataFrame({
    'Mercado':      resultado_eq['pesos_mercado']   * 100,
    'BL s/views':   resultado_eq['pesos_otimos']    * 100,
    'BL c/views RSI': resultado_views['pesos_otimos'] * 100,
}).round(2)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(df_plot))
w = 0.27
for i, col in enumerate(df_plot.columns):
    ax.bar(x + (i - 1) * w, df_plot[col], w, label=col, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(df_plot.index, rotation=30)
ax.set_title('Pesos: Mercado vs Black-Litterman', fontsize=13)
ax.set_ylabel('%')
ax.legend()
plt.tight_layout()
plt.show()

print('\nEstatísticas do portfólio BL com views:')
for k, v in resultado_views['estatisticas'].items():
    print(f'  {k}: {v}')